# Lexicon maintenance - gloss repair (propose -> review -> apply)

Durable, package-level driver for the phase-05.55 maintenance loop (phase 8
inherits this shape). It is a **thin caller**: every operation is a package
function in `lang_tools.lexicon.maintenance`; no logic lives here.

The loop:

1. **Propose** - `thin_gloss_worklist` finds the re-scoped `definition == lemma`
   residue (a gloss equal to the concept's *sole* member form) with its repair
   context; the `gloss_repair` chain drafts one proposal per entry; proposals
   land in a reviewable JSONL under the gitignored staging area.
2. **Review** - a human edits the JSONL: refine `proposed_definition` if needed
   and flip `status` to `accepted` (or `rejected`). Nothing applies unreviewed.
3. **Apply** - `apply_gloss_proposals` rewrites only the accepted glosses in the
   concepts Parquet, preserving every row's provenance tag and re-tagging the
   edited rows `llm`. Ids never change, so no other table is touched.
4. **Gate** - re-run the quality checks; the `definition == lemma` invariant
   should now read 0.

In [ ]:
from lang_tools.lexicon.ingestion.acquire import raw_dir
from lang_tools.lexicon.maintenance import apply_gloss_proposals
from lang_tools.lexicon.maintenance import GlossProposal
from lang_tools.lexicon.maintenance import read_proposals
from lang_tools.lexicon.maintenance import thin_gloss_worklist
from lang_tools.lexicon.maintenance import write_proposals
from lang_tools.params.lang_tools_params import get_lang_tools_params
from lang_tools.params.load_env import load_env

load_env()
data_fol = get_lang_tools_params().paths.data_fol
proposals_path = raw_dir(data_fol) / "staging" / "gloss_repair.jsonl"
proposals_path

## Propose

Build the worklist, then draft one proposal per entry with the `gloss_repair`
chain (grounded in the entry's member forms / English gloss / lexfile /
hypernym gloss, so it cannot invent meaning).

In [ ]:
worklist = thin_gloss_worklist(data_fol)
worklist

In [ ]:
from lang_tools.llm import build_gloss_repair_chain
from lang_tools.llm import GlossRepairInput
from llm_core.chat.config.openai import ChatOpenAIConfig

chain = build_gloss_repair_chain(ChatOpenAIConfig())

proposals = []
for entry in worklist:
    out = chain.invoke(
        GlossRepairInput(
            language=entry.language,
            current_definition=entry.definition,
            member_forms=[entry.member],
            english_definition=entry.english_definition,
            english_members=entry.english_members,
            lexfile=entry.lexfile,
            hypernym_definition=entry.hypernym_definition,
        )
    )
    proposals.append(
        GlossProposal(
            concept_id=entry.concept_id,
            language=entry.language,
            current_definition=entry.definition,
            proposed_definition=out.proposed_definition,
            rationale=out.rationale,
        )
    )

write_proposals(proposals, proposals_path)
proposals

## Review

Open the JSONL above, check each `proposed_definition`, refine it if needed,
and flip `status` to `accepted` (or `rejected`). The file is staging scratch -
never commit it.

## Apply (deliberate)

Uncomment after review. Only `accepted` rows apply; edited rows re-tag
`source=llm`; everything else is untouched.

In [ ]:
reviewed = read_proposals(proposals_path)
# apply_gloss_proposals(reviewed, data_fol=data_fol)
reviewed

## Gate

Re-run the quality checks (or the 05.4 report notebook); the
`definition == lemma` invariant should now read 0.

In [ ]:
from lang_tools.lexicon.quality import run_quality_checks

report = run_quality_checks(data_fol)
[(inv.name, inv.value, inv.passed) for inv in report.invariants]